# DL Baseline Comparison: 3-Fold Cross-Validation

This notebook adds two proper DL baselines to the cross-validation comparison:

1. **Q-Former v4** (full system): pretrained Q-Former + attn pool + gated residual + all v3 improvements
2. **Concat v3** (ablation): same pretrained system minus Q-Former fusion (mean pool instead)
3. **Detached MLP** (simpler DL): same pretrained checkpoint, bypasses Q-Former, uses `ic50_head_detached` (2-layer MLP on drug+cellline), basic training recipe (MSE loss, flat LR, no EMA/R-Drop)
4. **Simple MLP** (from scratch): no pretraining, raw embeddings → 3-layer MLP, standard training

### Why these baselines?
The Concat v3 baseline is **not a fair DL baseline** — it benefits from pretrained projectors, typed token embeddings, CellLineEncoder (cancer priors, FiLM tissue modulation, RNA fusion), and all v3 training improvements. We need baselines that isolate the value of:
- **Detached MLP**: Shows value of Q-Former fusion (same pretrained features, simpler head + no training tricks)
- **Simple MLP**: Shows value of entire pretraining framework (raw embeddings, no pretrained components)

In [1]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Subset
from tqdm import tqdm
import numpy as np
from scipy import stats
from collections import defaultdict
import json
from pathlib import Path
import copy

from gastro_transformer.config import GastroTransformerConfig
from gastro_transformer.model import ModalitySlotQFormer
from gastro_transformer.data import (
    DrugEmbeddingDataset,
    IC50Dataset,
)
from gastro_transformer.losses import compute_ic50_metrics

/opt/conda/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
# Configuration
DEVICE = 'cuda:0'
BATCH_SIZE = 256
N_FOLDS = 3
SEED = 42

# Detached MLP settings
DETACHED_EPOCHS = 10

# Simple MLP settings (more epochs since no pretraining warmstart)
SIMPLE_MLP_EPOCHS = 25
SIMPLE_MLP_LR = 1e-3

ROOT_DIR = '../'
CHECKPOINT_PATH = ROOT_DIR + 'saved_checkpoints/pretrained_clrna.pt'

OUTPUT_DIR = Path(ROOT_DIR + 'reports/cross_validation_v4')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Data paths
DRUG_EMBEDDINGS_CSV = ROOT_DIR + 'data/drug_embeddings.csv'
IC50_CSV = ROOT_DIR + 'data/ic50_data.csv'
CELLLINE_RNA_CSV = ROOT_DIR + 'data/processed/ccle_rna_for_ic50.csv'

print(f"Device: {DEVICE}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Detached MLP epochs: {DETACHED_EPOCHS}")
print(f"Simple MLP epochs: {SIMPLE_MLP_EPOCHS}, LR: {SIMPLE_MLP_LR}")

Device: cuda:0
Checkpoint: ../saved_checkpoints/pretrained_clrna.pt
Detached MLP epochs: 10
Simple MLP epochs: 25, LR: 0.001


In [3]:
# Load IC50 data
drug_dataset = DrugEmbeddingDataset(DRUG_EMBEDDINGS_CSV)
ic50_dataset = IC50Dataset(
    IC50_CSV,
    drug_dataset,
    rna_csv_path=CELLLINE_RNA_CSV,
    add_tissue_ids=True
)
print(f"Loaded IC50 dataset: {len(ic50_dataset)} samples")

# Dataset stats for Simple MLP
num_unique_celllines = len(np.unique(ic50_dataset.cellline_ids))
drug_dim = drug_dataset.embeddings.shape[1]  # 768
rna_dim = 256  # BulkRNABERT dimension
print(f"Unique cell-lines: {num_unique_celllines}")
print(f"Drug embedding dim: {drug_dim}")
print(f"RNA dim: {rna_dim}")

Loaded 234 drug embeddings from ../data/drug_embeddings.csv
Loaded 185294 IC50 entries from ../data/ic50_data.csv
After filtering to available drugs: 185294 entries
After removing duplicates: 173942 entries
IC50 values stored as-is (log_transform=False): range [-9.99, 12.90], std=2.774
Number of unique cell-lines: 998
Resolved tissue IDs for 998 unique cell-lines


../gastro_transformer/data.py:457: UserWarning: Found 11352 duplicate drug-cellline pairs. Keeping first occurrence of each pair.
  warnings.warn(


Loaded RNA embeddings for 592 cell-lines
RNA availability: 592/998 cell-lines have RNA, 406 missing
Loaded IC50 dataset: 173942 samples
Unique cell-lines: 998
Drug embedding dim: 768
RNA dim: 256


In [4]:
# Create cell-line-aware CV splits (identical to cross_validation.ipynb)
def create_cv_splits(ic50_dataset, n_folds=3, seed=42):
    cellline_ids = ic50_dataset.cellline_ids
    unique_celllines = np.unique(cellline_ids)

    np.random.seed(seed)
    np.random.shuffle(unique_celllines)

    fold_size = len(unique_celllines) // n_folds
    folds = []
    for i in range(n_folds):
        if i < n_folds - 1:
            test_cl = unique_celllines[i * fold_size:(i + 1) * fold_size]
        else:
            test_cl = unique_celllines[i * fold_size:]

        train_val_cl = np.array([c for c in unique_celllines if c not in test_cl])
        val_size = len(train_val_cl) // 4
        np.random.seed(seed + i)
        np.random.shuffle(train_val_cl)
        val_cl = train_val_cl[:val_size]
        train_cl = train_val_cl[val_size:]

        train_idx = np.where(np.isin(cellline_ids, train_cl))[0].tolist()
        val_idx = np.where(np.isin(cellline_ids, val_cl))[0].tolist()
        test_idx = np.where(np.isin(cellline_ids, test_cl))[0].tolist()

        folds.append((train_idx, val_idx, test_idx))

    return folds

cv_splits = create_cv_splits(ic50_dataset, N_FOLDS, SEED)
print(f"Dataset size: {len(ic50_dataset)}")
for i, (train_idx, val_idx, test_idx) in enumerate(cv_splits):
    print(f"Fold {i+1}: Train={len(train_idx)}, Val={len(val_idx)}, Test={len(test_idx)}")

Dataset size: 173942
Fold 1: Train=85830, Val=29184, Test=58928
Fold 2: Train=87667, Val=29101, Test=57174
Fold 3: Train=87536, Val=28566, Test=57840


In [5]:
# Shared evaluation function
def evaluate_on_test(model, test_loader, device, is_simple_mlp=False):
    """Evaluate model on test set."""
    model.eval()
    test_preds = []
    test_targets = []

    with torch.no_grad():
        for batch in test_loader:
            drug_embeds = batch["drug_embed"].to(device)
            ic50_targets = batch["ic50"].to(device)
            cellline_ids = batch["cellline_id"].to(device)

            if is_simple_mlp:
                rna_embed = batch.get("rna_embed")
                rna_available = batch.get("rna_available")
                if rna_embed is not None:
                    rna_embed = rna_embed.to(device)
                if rna_available is not None:
                    rna_available = rna_available.to(device)
                preds = model(drug_embeds, cellline_ids, rna_embed, rna_available)
            else:
                cancer_type_ids = batch.get("cancer_type_id")
                if cancer_type_ids is not None:
                    cancer_type_ids = cancer_type_ids.to(device)
                tissue_ids_batch = batch.get("tissue_id")
                if tissue_ids_batch is not None:
                    tissue_ids_batch = tissue_ids_batch.to(device)
                cellline_rna_embeds = batch.get("rna_embed")
                if cellline_rna_embeds is not None:
                    cellline_rna_embeds = cellline_rna_embeds.to(device)
                rna_available = batch.get("rna_available")
                if rna_available is not None:
                    rna_available = rna_available.to(device)

                outputs = model(
                    drug_embeds=drug_embeds,
                    cellline_ids=cellline_ids,
                    cancer_type_ids=cancer_type_ids,
                    tissue_ids=tissue_ids_batch,
                    cellline_rna_embeds=cellline_rna_embeds,
                    rna_available=rna_available,
                )
                preds = outputs["ic50_pred"]

            test_preds.extend(preds.cpu().numpy())
            test_targets.extend(ic50_targets.cpu().numpy())

    test_metrics = compute_ic50_metrics(
        torch.tensor(test_preds),
        torch.tensor(test_targets)
    )

    return {
        "predictions": test_preds,
        "targets": test_targets,
        "metrics": test_metrics
    }

print("Helper functions defined.")

Helper functions defined.


## Baseline 1: Detached MLP

Uses the same pretrained checkpoint (projectors + CellLineEncoder) but bypasses Q-Former fusion.
The `ic50_head_detached` is a 2-layer MLP that takes `cat(drug_proj, cellline_emb)` directly.

**Training recipe**: Simple MSE loss, flat LR (AdamW + CosineAnnealing), no EMA, no R-Drop, no warmup, no multi-task.

In [6]:
def train_fold_simple(model, train_loader, val_loader, device, epochs, lr=1e-4):
    """Train a single fold with simple recipe: MSE + AdamW + CosineAnnealing.
    No EMA, no R-Drop, no warmup, no multi-task, no differential LR."""
    scaler = GradScaler("cuda") if "cuda" in device else None
    device_type = "cuda" if "cuda" in device else "cpu"

    best_val_loss = float("inf")
    best_state = None

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        n_batches = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for batch in pbar:
            optimizer.zero_grad()

            drug_embeds = batch["drug_embed"].to(device)
            ic50_targets = batch["ic50"].to(device)
            cellline_ids = batch["cellline_id"].to(device)

            cancer_type_ids = batch.get("cancer_type_id")
            if cancer_type_ids is not None:
                cancer_type_ids = cancer_type_ids.to(device)
            tissue_ids_batch = batch.get("tissue_id")
            if tissue_ids_batch is not None:
                tissue_ids_batch = tissue_ids_batch.to(device)
            cellline_rna_embeds = batch.get("rna_embed")
            if cellline_rna_embeds is not None:
                cellline_rna_embeds = cellline_rna_embeds.to(device)
            rna_available = batch.get("rna_available")
            if rna_available is not None:
                rna_available = rna_available.to(device)

            with autocast(device_type, enabled=scaler is not None):
                outputs = model(
                    drug_embeds=drug_embeds,
                    cellline_ids=cellline_ids,
                    cancer_type_ids=cancer_type_ids,
                    tissue_ids=tissue_ids_batch,
                    cellline_rna_embeds=cellline_rna_embeds,
                    rna_available=rna_available,
                )
                loss = F.mse_loss(outputs["ic50_pred"], ic50_targets)

            if scaler:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            train_loss += loss.item()
            n_batches += 1
            pbar.set_postfix({"loss": f"{train_loss / n_batches:.4f}"})

        # Validation
        model.eval()
        val_loss = 0
        val_n = 0
        with torch.no_grad():
            for batch in val_loader:
                drug_embeds = batch["drug_embed"].to(device)
                ic50_targets = batch["ic50"].to(device)
                cellline_ids = batch["cellline_id"].to(device)
                cancer_type_ids = batch.get("cancer_type_id")
                if cancer_type_ids is not None:
                    cancer_type_ids = cancer_type_ids.to(device)
                tissue_ids_batch = batch.get("tissue_id")
                if tissue_ids_batch is not None:
                    tissue_ids_batch = tissue_ids_batch.to(device)
                cellline_rna_embeds = batch.get("rna_embed")
                if cellline_rna_embeds is not None:
                    cellline_rna_embeds = cellline_rna_embeds.to(device)
                rna_available = batch.get("rna_available")
                if rna_available is not None:
                    rna_available = rna_available.to(device)

                outputs = model(
                    drug_embeds=drug_embeds,
                    cellline_ids=cellline_ids,
                    cancer_type_ids=cancer_type_ids,
                    tissue_ids=tissue_ids_batch,
                    cellline_rna_embeds=cellline_rna_embeds,
                    rna_available=rna_available,
                )
                val_loss += F.mse_loss(outputs["ic50_pred"], ic50_targets).item()
                val_n += 1

        avg_val_loss = val_loss / max(val_n, 1)
        print(f"Epoch {epoch+1}/{epochs} - Train: {train_loss/n_batches:.4f}, Val: {avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_state = copy.deepcopy(model.state_dict())

        scheduler.step()

    if best_state is not None:
        model.load_state_dict(best_state)


def run_cv_detached(ic50_dataset, cv_splits, device, epochs, checkpoint_path):
    """Run CV with detached MLP (bypasses Q-Former, simple training)."""
    print(f"\n{'='*60}")
    print(f"Running CV for: Detached MLP")
    print(f"use_qformer_for_ic50=False, MSE loss, flat LR, no EMA/R-Drop")
    print(f"Epochs: {epochs}")
    print(f"{'='*60}")

    fold_metrics = []
    all_preds = []
    all_targets = []

    for fold_idx, (train_idx, val_idx, test_idx) in enumerate(cv_splits):
        print(f"\n--- Fold {fold_idx + 1}/{len(cv_splits)} ---")

        train_loader = DataLoader(Subset(ic50_dataset, train_idx), batch_size=BATCH_SIZE,
                                  shuffle=True, num_workers=4, persistent_workers=True)
        val_loader = DataLoader(Subset(ic50_dataset, val_idx), batch_size=BATCH_SIZE,
                                shuffle=False, num_workers=4, persistent_workers=True)
        test_loader = DataLoader(Subset(ic50_dataset, test_idx), batch_size=BATCH_SIZE,
                                 shuffle=False, num_workers=4, persistent_workers=True)

        # Create model with detached config
        fold_config = GastroTransformerConfig()
        fold_config.num_query_tokens = 32
        fold_config.qformer_layers = 6
        fold_config.use_qformer = True  # Need Q-Former in model for checkpoint loading
        fold_config.use_qformer_for_ic50 = False  # But bypass it for IC50
        fold_config.batch_size = BATCH_SIZE
        fold_config.num_workers = 4
        fold_config.persistent_workers = True

        model = ModalitySlotQFormer(fold_config).to(device)

        # Load pretrained checkpoint
        if checkpoint_path and Path(checkpoint_path).exists():
            print(f"Loading checkpoint: {checkpoint_path}")
            checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
            model.load_state_dict(checkpoint["model_state_dict"], strict=False)

        # Train with simple recipe
        train_fold_simple(model, train_loader, val_loader, device, epochs, lr=1e-4)

        # Evaluate
        test_results = evaluate_on_test(model, test_loader, device)

        print(f"Fold {fold_idx + 1} - R2: {test_results['metrics']['r2']:.4f}, "
              f"Pearson: {test_results['metrics']['pearson_r']:.4f}, "
              f"Spearman: {test_results['metrics'].get('spearman_r', 0):.4f}")

        fold_metrics.append(test_results["metrics"])
        all_preds.extend(test_results["predictions"])
        all_targets.extend(test_results["targets"])

        del model
        torch.cuda.empty_cache()

    return {"fold_metrics": fold_metrics, "all_preds": all_preds, "all_targets": all_targets}

print("Detached MLP functions defined.")

Detached MLP functions defined.


## Baseline 2: Simple MLP from Scratch

No pretrained components. Raw embeddings → 3-layer MLP → IC50.

- **Drug**: raw ChemBERT embeddings (768d)
- **Cell-line**: learnable `nn.Embedding` (768d)
- **RNA**: raw BulkRNABERT embeddings (256d), zero-filled when missing
- **Input**: `cat(drug, cellline, rna)` = 1792d
- No cancer type priors, no FiLM tissue modulation, no RNA fusion gate

In [7]:
class SimpleMLP(nn.Module):
    """Simple MLP baseline: raw concat of drug + cellline + RNA → 3-layer MLP → IC50.
    No pretrained components, no cancer priors, no FiLM, no RNA fusion gate."""

    def __init__(self, drug_dim=768, cellline_embed_dim=768, rna_dim=256,
                 num_celllines=1200, hidden_dim=1024, dropout=0.1):
        super().__init__()
        self.cellline_embed = nn.Embedding(num_celllines, cellline_embed_dim)
        self.rna_dim = rna_dim

        input_dim = drug_dim + cellline_embed_dim + rna_dim  # 768 + 768 + 256 = 1792
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, drug_embeds, cellline_ids, rna_embeds=None, rna_available=None):
        """Forward pass.
        Args:
            drug_embeds: [B, 768] raw ChemBERT drug embeddings
            cellline_ids: [B] integer cell-line IDs
            rna_embeds: [B, 256] raw RNA embeddings (may be zero-filled)
            rna_available: [B] bool mask (True where RNA exists)
        Returns:
            IC50 predictions [B]
        """
        # Clamp cell-line IDs to valid range
        max_idx = self.cellline_embed.num_embeddings - 1
        safe_ids = cellline_ids.clamp(0, max_idx)
        cl_emb = self.cellline_embed(safe_ids)  # [B, 768]

        # RNA: use raw embeddings, zero-fill where missing
        if rna_embeds is not None:
            rna = rna_embeds
            if rna_available is not None:
                # Zero out RNA for samples without data
                rna = rna * rna_available.unsqueeze(-1).float()
        else:
            rna = torch.zeros(drug_embeds.shape[0], self.rna_dim, device=drug_embeds.device)

        x = torch.cat([drug_embeds, cl_emb, rna], dim=-1)  # [B, 1792]
        return self.mlp(x).squeeze(-1)  # [B]


def train_fold_simple_mlp(model, train_loader, val_loader, device, epochs, lr):
    """Train SimpleMLP with standard recipe: MSE + AdamW + CosineAnnealing."""
    scaler = GradScaler("cuda") if "cuda" in device else None
    device_type = "cuda" if "cuda" in device else "cpu"

    best_val_loss = float("inf")
    best_state = None

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        n_batches = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for batch in pbar:
            optimizer.zero_grad()

            drug_embeds = batch["drug_embed"].to(device)
            ic50_targets = batch["ic50"].to(device)
            cellline_ids = batch["cellline_id"].to(device)
            rna_embed = batch.get("rna_embed")
            if rna_embed is not None:
                rna_embed = rna_embed.to(device)
            rna_available = batch.get("rna_available")
            if rna_available is not None:
                rna_available = rna_available.to(device)

            with autocast(device_type, enabled=scaler is not None):
                preds = model(drug_embeds, cellline_ids, rna_embed, rna_available)
                loss = F.mse_loss(preds, ic50_targets)

            if scaler:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            train_loss += loss.item()
            n_batches += 1
            pbar.set_postfix({"loss": f"{train_loss / n_batches:.4f}"})

        # Validation
        model.eval()
        val_loss = 0
        val_n = 0
        with torch.no_grad():
            for batch in val_loader:
                drug_embeds = batch["drug_embed"].to(device)
                ic50_targets = batch["ic50"].to(device)
                cellline_ids = batch["cellline_id"].to(device)
                rna_embed = batch.get("rna_embed")
                if rna_embed is not None:
                    rna_embed = rna_embed.to(device)
                rna_available = batch.get("rna_available")
                if rna_available is not None:
                    rna_available = rna_available.to(device)

                preds = model(drug_embeds, cellline_ids, rna_embed, rna_available)
                val_loss += F.mse_loss(preds, ic50_targets).item()
                val_n += 1

        avg_val_loss = val_loss / max(val_n, 1)
        print(f"Epoch {epoch+1}/{epochs} - Train: {train_loss/n_batches:.4f}, Val: {avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_state = copy.deepcopy(model.state_dict())

        scheduler.step()

    if best_state is not None:
        model.load_state_dict(best_state)


def run_cv_simple_mlp(ic50_dataset, cv_splits, device, epochs, lr, num_celllines):
    """Run CV with SimpleMLP (no pretraining, raw embeddings)."""
    print(f"\n{'='*60}")
    print(f"Running CV for: Simple MLP (from scratch)")
    print(f"No pretraining, raw concat, MSE loss, AdamW(lr={lr})")
    print(f"Epochs: {epochs}")
    print(f"{'='*60}")

    fold_metrics = []
    all_preds = []
    all_targets = []

    for fold_idx, (train_idx, val_idx, test_idx) in enumerate(cv_splits):
        print(f"\n--- Fold {fold_idx + 1}/{len(cv_splits)} ---")

        train_loader = DataLoader(Subset(ic50_dataset, train_idx), batch_size=BATCH_SIZE,
                                  shuffle=True, num_workers=4, persistent_workers=True)
        val_loader = DataLoader(Subset(ic50_dataset, val_idx), batch_size=BATCH_SIZE,
                                shuffle=False, num_workers=4, persistent_workers=True)
        test_loader = DataLoader(Subset(ic50_dataset, test_idx), batch_size=BATCH_SIZE,
                                 shuffle=False, num_workers=4, persistent_workers=True)

        # Fresh model each fold (no pretraining)
        model = SimpleMLP(
            drug_dim=768,
            cellline_embed_dim=768,
            rna_dim=256,
            num_celllines=num_celllines,
            hidden_dim=1024,
            dropout=0.1
        ).to(device)

        param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
        if fold_idx == 0:
            print(f"SimpleMLP params: {param_count:,}")

        train_fold_simple_mlp(model, train_loader, val_loader, device, epochs, lr)

        # Evaluate
        test_results = evaluate_on_test(model, test_loader, device, is_simple_mlp=True)

        print(f"Fold {fold_idx + 1} - R2: {test_results['metrics']['r2']:.4f}, "
              f"Pearson: {test_results['metrics']['pearson_r']:.4f}, "
              f"Spearman: {test_results['metrics'].get('spearman_r', 0):.4f}")

        fold_metrics.append(test_results["metrics"])
        all_preds.extend(test_results["predictions"])
        all_targets.extend(test_results["targets"])

        del model
        torch.cuda.empty_cache()

    return {"fold_metrics": fold_metrics, "all_preds": all_preds, "all_targets": all_targets}

print("Simple MLP functions defined.")

Simple MLP functions defined.


## Run Both Baselines

In [8]:
# Run Detached MLP CV
detached_results = run_cv_detached(
    ic50_dataset=ic50_dataset,
    cv_splits=cv_splits,
    device=DEVICE,
    epochs=DETACHED_EPOCHS,
    checkpoint_path=CHECKPOINT_PATH
)


Running CV for: Detached MLP
use_qformer_for_ic50=False, MSE loss, flat LR, no EMA/R-Drop
Epochs: 10

--- Fold 1/3 ---
Loading checkpoint: ../saved_checkpoints/pretrained_clrna.pt


Epoch 1/10: 100%|█████████████████████████████████████████████████| 336/336 [00:38<00:00,  8.65it/s, loss=3.2072]


Epoch 1/10 - Train: 3.2072, Val: 2.2513


Epoch 2/10: 100%|█████████████████████████████████████████████████| 336/336 [00:33<00:00,  9.94it/s, loss=1.8509]


Epoch 2/10 - Train: 1.8509, Val: 2.3625


...

Epoch 8/10 - Train: 1.1782, Val: 2.0527


Epoch 9/10: 100%|█████████████████████████████████████████████████| 342/342 [00:34<00:00,  9.95it/s, loss=1.1215]


Epoch 9/10 - Train: 1.1215, Val: 2.0625


Epoch 10/10: 100%|████████████████████████████████████████████████| 342/342 [00:35<00:00,  9.71it/s, loss=1.0847]


Epoch 10/10 - Train: 1.0847, Val: 2.0765
Fold 3 - R2: 0.7456, Pearson: 0.8638, Spearman: 0.8310


In [9]:
# Run Simple MLP CV
simple_mlp_results = run_cv_simple_mlp(
    ic50_dataset=ic50_dataset,
    cv_splits=cv_splits,
    device=DEVICE,
    epochs=SIMPLE_MLP_EPOCHS,
    lr=SIMPLE_MLP_LR,
    num_celllines=num_unique_celllines + 100  # Safety margin for cell-line IDs
)


Running CV for: Simple MLP (from scratch)
No pretraining, raw concat, MSE loss, AdamW(lr=0.001)
Epochs: 25

--- Fold 1/3 ---
SimpleMLP params: 3,206,657


Epoch 1/25: 100%|█████████████████████████████████████████████████| 336/336 [00:04<00:00, 72.09it/s, loss=4.5832]


Epoch 1/25 - Train: 4.5832, Val: 3.3787


Epoch 2/25: 100%|█████████████████████████████████████████████████| 336/336 [00:04<00:00, 73.94it/s, loss=2.5395]


Epoch 2/25 - Train: 2.5395, Val: 2.6669


...

Epoch 23/25 - Train: 0.4050, Val: 2.7722


Epoch 24/25: 100%|███████████████████████████████████████████████| 342/342 [00:01<00:00, 202.47it/s, loss=0.3955]


Epoch 24/25 - Train: 0.3955, Val: 2.7743


Epoch 25/25: 100%|███████████████████████████████████████████████| 342/342 [00:01<00:00, 184.12it/s, loss=0.3900]


Epoch 25/25 - Train: 0.3900, Val: 2.7622
Fold 3 - R2: 0.6722, Pearson: 0.8276, Spearman: 0.7831


## Results Comparison

In [10]:
def compute_cv_summary(fold_metrics):
    """Compute mean, std, 95% CI for fold metrics."""
    summary = {}
    for key in ['r2', 'pearson_r', 'spearman_r', 'rmse', 'mae']:
        values = [fm[key] for fm in fold_metrics if key in fm]
        if values:
            mean = np.mean(values)
            std = np.std(values, ddof=1) if len(values) > 1 else 0.0
            ci95 = 1.96 * std / np.sqrt(len(values)) if len(values) > 1 else 0.0
            summary[key] = mean
            summary[f'{key}_std'] = std
            summary[f'{key}_ci95'] = ci95
    return summary


# Previous results from cross_validation.ipynb (loaded from saved JSON)
prev_results_path = OUTPUT_DIR / 'qformer_v4_results.json'
prev_results_v3_path = Path(ROOT_DIR + 'reports/cross_validation_v3/cross_validation_results.json')

# Load previous CV results
prior_models = {}

if prev_results_v3_path.exists():
    with open(prev_results_v3_path) as f:
        v3_data = json.load(f)
    if 'qformer_v3' in v3_data:
        prior_models['Q-Former v3'] = v3_data['qformer_v3']['fold_metrics']
    if 'concat_baseline_v3' in v3_data:
        prior_models['Concat v3'] = v3_data['concat_baseline_v3']['fold_metrics']

if prev_results_path.exists():
    with open(prev_results_path) as f:
        v4_data = json.load(f)
    if 'fold_metrics' in v4_data:
        prior_models['Q-Former v4'] = v4_data['fold_metrics']

# Add new results
all_models = {}
all_models.update(prior_models)
all_models['Detached MLP'] = detached_results['fold_metrics']
all_models['Simple MLP'] = simple_mlp_results['fold_metrics']

# Print comparison table
print(f"\n{'='*90}")
print(f"{'Model':<20} {'R²':>8} {'±std':>8} {'Pearson':>8} {'±std':>8} {'Spearman':>8} {'RMSE':>8} {'MAE':>8}")
print(f"{'='*90}")

summaries = {}
for name, fold_metrics in all_models.items():
    s = compute_cv_summary(fold_metrics)
    summaries[name] = s
    print(f"{name:<20} {s['r2']:>8.4f} {s.get('r2_std',0):>8.4f} "
          f"{s['pearson_r']:>8.4f} {s.get('pearson_r_std',0):>8.4f} "
          f"{s.get('spearman_r', 0):>8.4f} {s.get('rmse', 0):>8.4f} {s.get('mae', 0):>8.4f}")

print(f"{'='*90}")


Model                      R²     ±std  Pearson     ±std Spearman     RMSE      MAE
Q-Former v3            0.7420   0.0053   0.8621   0.0027   0.8312   1.4088   1.0438
Concat v3              0.7428   0.0050   0.8621   0.0029   0.8317   1.4065   1.0469
Q-Former v4            0.7428   0.0054   0.8627   0.0027   0.8322   1.4065   1.0423
Detached MLP           0.7350   0.0092   0.8586   0.0048   0.8261   1.4276   1.0668
Simple MLP             0.6761   0.0036   0.8295   0.0016   0.7900   1.5784   1.1855


In [11]:
# Save results
dl_baselines_results = {
    'config': {
        'n_folds': N_FOLDS,
        'seed': SEED,
        'detached_epochs': DETACHED_EPOCHS,
        'simple_mlp_epochs': SIMPLE_MLP_EPOCHS,
        'simple_mlp_lr': SIMPLE_MLP_LR,
        'batch_size': BATCH_SIZE,
        'checkpoint': CHECKPOINT_PATH,
    },
    'detached_mlp': {
        'fold_metrics': detached_results['fold_metrics'],
        'average_metrics': summaries.get('Detached MLP', {}),
        'description': 'Pretrained checkpoint, bypasses Q-Former, ic50_head_detached (2-layer MLP), MSE loss, flat LR, no EMA/R-Drop/warmup/multi-task',
    },
    'simple_mlp': {
        'fold_metrics': simple_mlp_results['fold_metrics'],
        'average_metrics': summaries.get('Simple MLP', {}),
        'description': 'No pretraining, raw drug+cellline_embed+RNA concat, 3-layer MLP, MSE loss, AdamW, CosineAnnealing',
    },
}

# Add prior results for complete comparison
for name, fold_metrics in prior_models.items():
    key = name.lower().replace(' ', '_').replace('-', '')
    dl_baselines_results[key] = {
        'fold_metrics': fold_metrics,
        'average_metrics': summaries.get(name, {}),
    }

output_path = OUTPUT_DIR / 'dl_baselines_results.json'
with open(output_path, 'w') as f:
    json.dump(dl_baselines_results, f, indent=2, default=float)
print(f"Results saved to {output_path}")

Results saved to ../reports/cross_validation_v4/dl_baselines_results.json


In [12]:
# Print fold-wise details
print("\nFold-wise R² breakdown:")
print(f"{'Model':<20} {'Fold 1':>8} {'Fold 2':>8} {'Fold 3':>8} {'Mean':>8}")
print("-" * 60)
for name, fold_metrics in all_models.items():
    r2s = [fm['r2'] for fm in fold_metrics]
    row = f"{name:<20}"
    for r2 in r2s:
        row += f" {r2:>8.4f}"
    row += f" {np.mean(r2s):>8.4f}"
    print(row)

print("\n--- Narrative ---")
if 'Q-Former v4' in summaries and 'Simple MLP' in summaries:
    delta = summaries['Q-Former v4']['r2'] - summaries['Simple MLP']['r2']
    print(f"Full system (Q-Former v4) vs Simple MLP: +{delta:.4f} R² ({delta*100:.1f}% absolute)")
if 'Q-Former v4' in summaries and 'Detached MLP' in summaries:
    delta = summaries['Q-Former v4']['r2'] - summaries['Detached MLP']['r2']
    print(f"Full system (Q-Former v4) vs Detached MLP: +{delta:.4f} R² ({delta*100:.1f}% absolute)")
if 'Detached MLP' in summaries and 'Simple MLP' in summaries:
    delta = summaries['Detached MLP']['r2'] - summaries['Simple MLP']['r2']
    print(f"Pretrained features value (Detached vs Simple): +{delta:.4f} R²")
if 'Q-Former v4' in summaries and 'Detached MLP' in summaries:
    delta = summaries['Q-Former v4']['r2'] - summaries['Detached MLP']['r2']
    print(f"Q-Former fusion + training tricks value: +{delta:.4f} R²")


Fold-wise R² breakdown:
Model                  Fold 1   Fold 2   Fold 3     Mean
------------------------------------------------------------
Q-Former v3            0.7393   0.7385   0.7481   0.7420
Concat v3              0.7409   0.7390   0.7485   0.7428
Q-Former v4            0.7393   0.7400   0.7491   0.7428
Detached MLP           0.7296   0.7298   0.7456   0.7350
Simple MLP             0.6769   0.6793   0.6722   0.6761

--- Narrative ---
Full system (Q-Former v4) vs Simple MLP: +0.0667 R² (6.7% absolute)
Full system (Q-Former v4) vs Detached MLP: +0.0078 R² (0.8% absolute)
Pretrained features value (Detached vs Simple): +0.0589 R²
Q-Former fusion + training tricks value: +0.0078 R²
